## Profiles, overrides & project name

Three features for running the *same* stack differently across dev, CI, and prod — and one gotcha that bites everyone.

**Profiles — opt-in services.** Tag a service with `profiles:` and it's **skipped unless you enable that profile** — perfect for a debug sidecar, a seed-data job, or a load tester that shouldn't pollute a normal `up`:

```yaml
services:
  web:      { image: myorg/web }
  debugger: { image: nicolaka/netshoot, profiles: [debug], command: sleep infinity }
```

`docker compose up` starts only `web`; `docker compose --profile debug up` adds `debugger`. A service with **no** `profiles:` always runs; one with a profile runs only when that profile is on.

**Overrides — layering files.** A `compose.override.yaml` sitting next to `compose.yaml` is **auto-loaded and merged**, override winning. The convention: `compose.yaml` is the production-ish base, `compose.override.yaml` is dev-only and gitignored, so a new dev clones, runs `up`, and gets the dev-friendly stack. For more layers, stack `-f` explicitly — `docker compose -f compose.yaml -f compose.prod.yaml up` — later files winning. Merge rule to remember: **scalars replace, lists append** (so `ports:`/`volumes:` can accumulate unexpected duplicates).

**Project name — the collision gotcha.** Compose namespaces everything by a **project name** (default: the directory basename): containers `<project>-<service>-N`, networks `<project>_default`, volumes `<project>_<name>`. So if you clone the same repo into two folders both named `myapp` and `up` each, **they fight over identical names** — the second errors or steals the first's containers. Fix: a unique name per checkout — `name: my-app` in the file, `docker compose -p myapp-scratch up`, or `COMPOSE_PROJECT_NAME` in `.env`.